# 🏦 1.3 užduotis (a): Bank Marketing – TensorFlow neuroninis tinklas

Perdarytas dėstytojo notebook'as **„AdventureWorks DW using TensorFlow Neural Network“** su kitais verslo duomenimis.

**Duomenys:** [UCI Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing) (taip pat [Kaggle](https://www.kaggle.com/datasets/henriqueyamahata/bank-marketing)), licencija CC BY 4.0.
Portugalijos banko tiesioginės rinkodaros kampanija: banko darbuotojai skambino klientams ir siūlė terminuotą indėlį.
45 211 klientų, 16 požymių, tikslas `y` – ar klientas sutiko (*yes*/*no*). Kopija saugoma [`data/bank_marketing/`](https://github.com/edvakasas/Tensorflow/tree/main/data/bank_marketing).

Tai tiesioginis vTargetMail analogas: ten – „ar klientas pirks dviratį?“, čia – „ar klientas sutiks su indėliu?“.

*[Edvin Gladkich, Nojus Sadaunykas]*

## Kaip veikia originalus notebook'as (AdventureWorksDW_NN)

| Žingsnis | Ką daro originalas | Kodėl |
|---|---|---|
| 1. Duomenys | Iš GitHub įkelia `vTargetMail.csv` (AdventureWorks DW rodinys: klientai ir požymis `BikeBuyer`). | Verslo klausimas – kuriems klientams siųsti reklamą. |
| 2. Valymas | Pašalina identifikatorius ir asmens duomenis (vardai, el. paštas, telefonas, adresas, kitų kalbų stulpeliai), `BikeBuyer` pervadina į `target`. | Šie stulpeliai neturi prognozinės vertės arba yra unikalūs kiekvienam klientui – modelis juos tik „įsimintų“. |
| 3. Aibės | Sumaišo ir padalija 80:10:10 į train / validation / test. | Validavimo aibė – epochų stebėjimui, testavimo – galutiniam įvertinimui. |
| 4. `df_to_dataset` | Pandas lentelę paverčia `tf.data.Dataset`: požymių žodynas + etiketės, sumaišymas, paketai (batch 256), `prefetch`. | Efektyvus duomenų tiekimas Keras modeliui. |
| 5. Kategoriniai požymiai | `StringLookup` / `IntegerLookup` paverčia reikšmes indeksais, `CategoryEncoding` – vektoriumi (one-hot). Žodynas išmokstamas iš train (`adapt`). | Neuroninis tinklas dirba tik su skaičiais. |
| 6. Skaitiniai požymiai | `Normalization` sluoksnis: (x − vidurkis) / std, statistika iš train. | Skirtingų mastelių požymiai (pajamos vs. vaikų skaičius) neturi užgožti vienas kito. |
| 7. Modelis | Keras Functional API: visi užkoduoti požymiai sujungiami (`concatenate`) → Dense 64 (ReLU) → Dropout 0,2 → Dense 32 → Dropout 0,3 → Dense 1 (logit). Nuostolis – `BinaryCrossentropy(from_logits=True)`, optimizatorius Adam. | Dvejetainė klasifikacija; Dropout mažina persimokymą. |
| 8. Mokymas | 30 epochų, stebima validation accuracy, vėliau – TensorBoard. | Matyti, ar modelis mokosi ir ar nepersimoko. |
| 9. Įvertinimas ir naudojimas | `evaluate` testavimo aibėje, `save` / `load_model`, tikimybė vienam naujam klientui per `sigmoid(logit)`. | Modelis paruoštas naudoti realiam klientui. |

**Pastebėjimas apie originalo kodą:** `df_to_dataset` funkcijoje požymių žodynas kuriamas iš `dataframe.items()`, o ne iš kopijos `df`, iš kurios jau pašalinta etiketė. Todėl `target` stulpelis patenka ir tarp įvesčių. Modelis jo nenaudoja (tokios `Input` nėra), bet tai klaida, todėl mūsų versijoje ji ištaisyta.

## Ką keičiame su Bank Marketing duomenimis
- **Nutekėjimas:** pašaliname `duration` (žinoma tik po skambučio).
- **Nesubalansuotos klasės:** tik 11,7 % „yes“ (vTargetMail – apie pusė pirkėjų), todėl be accuracy matuojame **AUC, precision, recall** ir naudojame **klasių svorius**.
- Išvestis su `sigmoid` (tikimybė), kad metrikos būtų tiesiogiai interpretuojamos; mokymas su `EarlyStopping`.
- Vietoj TensorBoard – mokymo kreivės tiesiai notebook'e.

In [ ]:
#@title 📥 STEP 1: Duomenų įkėlimas iš GitHub
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/edvakasas/Tensorflow/main/data/bank_marketing/bank-full.csv"
df_all_col = pd.read_csv(DATA_URL, sep=";")
print("The dataframe has {} rows and {} columns.".format(df_all_col.shape[0], df_all_col.shape[1]))
display(df_all_col.head())

In [ ]:
#@title 🔎 STEP 2: Tiriamoji analizė – tikslo pasiskirstymas
import matplotlib.pyplot as plt

share = df_all_col["y"].value_counts(normalize=True)
print(share.round(3).to_string())
print(f"\nJei visada spėtume „no“, tikslumas būtų {share['no']:.1%} – todėl vien accuracy čia nepakanka.")

yes_rate = lambda col: df_all_col.groupby(col)["y"].apply(lambda s: (s == "yes").mean())
months = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
share.plot.bar(ax=axes[0], title="Tikslas y", color=["tab:grey", "tab:green"], rot=0)
yes_rate("job").sort_values().plot.barh(ax=axes[1], title="Sutikusiųjų dalis pagal profesiją")
yes_rate("month").reindex(months).plot.bar(ax=axes[2], title="Sutikusiųjų dalis pagal mėnesį", rot=0)
plt.tight_layout()
plt.show()

In [ ]:
#@title 🧹 STEP 3: Paruošimas – nutekėjimo pašalinimas ir tikslas
df = df_all_col.copy()

# duration – pokalbio trukmė. Ji žinoma tik PO skambučio, kai atsakymas jau aiškus,
# todėl modelis, išmokęs iš jos, realiai prognozuoti negalėtų (duomenų nutekėjimas).
df.drop(columns=["duration"], inplace=True)
df["target"] = (df.pop("y") == "yes").astype("int64")  # originale: BikeBuyer → target

CATEGORICAL = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome"]
NUMERIC = ["age", "balance", "day", "campaign", "pdays", "previous"]
df[CATEGORICAL] = df[CATEGORICAL].astype(object)
df[NUMERIC] = df[NUMERIC].astype("float32")
print(df.iloc[0])

In [ ]:
#@title ✂️ STEP 4: Mokymo, validavimo ir testavimo aibės (80:10:10)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # sumaišome; seed – kad rezultatai kartotųsi
n = len(df)
train = df.iloc[: int(0.8 * n)]
validation = df.iloc[int(0.8 * n): int(0.9 * n)]
test = df.iloc[int(0.9 * n):]  # ta pati testavimo aibė naudojama ir Decision Forests notebook'e
print(len(train), "training examples")
print(len(validation), "validation examples")
print(len(test), "test examples")
print(f"„yes“ dalis: train {train.target.mean():.3f}, validation {validation.target.mean():.3f}, test {test.target.mean():.3f}")

## 5 žingsnis – `tf.data` duomenų srautas (ištaisyta originalo klaida)

In [ ]:
#@title 🔁 STEP 5: df_to_dataset
import numpy as np
import tensorflow as tf

def df_to_dataset(dataframe, shuffle=True, batch_size=256):
    df = dataframe.copy()
    labels = df.pop("target")
    features = {key: value.to_numpy()[:, tf.newaxis] for key, value in df.items()}  # originale – dataframe.items()
    ds = tf.data.Dataset.from_tensor_slices((features, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=42)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = df_to_dataset(train)
validation_ds = df_to_dataset(validation, shuffle=False)
test_ds = df_to_dataset(test, shuffle=False)
print("TensorFlow", tf.__version__)

## 6 žingsnis – požymių kodavimas Keras sluoksniais (kaip originale)

In [ ]:
#@title 🧩 STEP 6: Kategorinių ir skaitinių požymių sluoksniai
all_inputs = {}
encoded_features = []

def get_category_encoding_layer(name, dataset, dtype, max_tokens=None):
    index = tf.keras.layers.StringLookup(max_tokens=max_tokens) if dtype == "string" \
        else tf.keras.layers.IntegerLookup(max_tokens=max_tokens)
    index.adapt(dataset.map(lambda x, y: x[name]))  # žodynas – tik iš mokymo aibės
    encoder = tf.keras.layers.CategoryEncoding(num_tokens=index.vocabulary_size())
    return lambda feature: encoder(index(feature))

def get_normalization_layer(name, dataset):
    normalizer = tf.keras.layers.Normalization(axis=None)
    normalizer.adapt(dataset.map(lambda x, y: x[name]))  # vidurkis ir dispersija – tik iš mokymo aibės
    return normalizer

for header in CATEGORICAL:
    col = tf.keras.Input(shape=(1,), name=header, dtype="string")
    all_inputs[header] = col
    encoded_features.append(get_category_encoding_layer(header, train_ds, "string")(col))

for header in NUMERIC:
    col = tf.keras.Input(shape=(1,), name=header)
    all_inputs[header] = col
    encoded_features.append(get_normalization_layer(header, train_ds)(col))

all_features = tf.keras.layers.concatenate(encoded_features)
print("Įvesties vektoriaus ilgis po kodavimo:", all_features.shape[-1])

## 7 žingsnis – modelis
Ta pati architektūra kaip originale (64 → 32 → 1 su Dropout), tik paskutinis sluoksnis su `sigmoid`, o metrikos papildytos AUC, precision ir recall.
Mokome du variantus: **A** – kaip originale, **B** – su klasių svoriais (retesnė klasė „yes“ gauna didesnį svorį nuostolio funkcijoje).

In [ ]:
#@title 🧠 STEP 7: Modelis ir mokymas (A – be svorių, B – su klasių svoriais)
def build_model():
    x = tf.keras.layers.Dense(64, activation="relu")(all_features)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Dense(32, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    output = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    model = tf.keras.Model(all_inputs, output)
    model.compile(optimizer="adam", loss="binary_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
                           tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")])
    return model

early_stop = lambda: tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True)

tf.keras.utils.set_random_seed(42)
model_a = build_model()
history_a = model_a.fit(train_ds, validation_data=validation_ds, epochs=30, callbacks=[early_stop()], verbose=0)

neg, pos = np.bincount(train["target"])
class_weight = {0: 1.0, 1: neg / pos}
print(f"Klasių svoriai: {class_weight}")
tf.keras.utils.set_random_seed(42)
model_b = build_model()
history_b = model_b.fit(train_ds, validation_data=validation_ds, epochs=30, callbacks=[early_stop()],
                        class_weight=class_weight, verbose=0)
print(f"A: {len(history_a.history['loss'])} epochų, B: {len(history_b.history['loss'])} epochų (EarlyStopping)")

In [ ]:
#@title 📈 STEP 8: Mokymo kreivės (vietoj TensorBoard)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name, h in [("A", history_a), ("B (klasių svoriai)", history_b)]:
    axes[0].plot(h.history["loss"], label=f"{name} train")
    axes[0].plot(h.history["val_loss"], "--", label=f"{name} validation")
    axes[1].plot(h.history["val_auc"], label=f"{name} validation AUC")
axes[0].set(title="Nuostolis", xlabel="epocha")
axes[1].set(title="Validation AUC", xlabel="epocha")
axes[0].legend(); axes[1].legend()
plt.tight_layout()
plt.show()

## 9 žingsnis – įvertinimas testavimo aibėje
Lyginame su naiviu „visada no“ spėjimu. **Recall** – kokią dalį tikrų „yes“ klientų modelis rado; **precision** – kokia dalis modelio pasirinktų klientų iš tiesų sutiko.
Rinkodarai dažnai svarbesnis recall: geriau paskambinti keliems papildomiems klientams, nei praleisti norinčius.

In [ ]:
#@title 🧪 STEP 9: Testavimo rezultatai ir klaidų matrica
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, ConfusionMatrixDisplay

y_test = test["target"].to_numpy()
rows, probs = [], {}
rows.append({"Modelis": "Visada „no“", "Accuracy": accuracy_score(y_test, np.zeros_like(y_test)), "AUC": 0.5, "Precision": 0.0, "Recall": 0.0})
for name, model in [("NN A", model_a), ("NN B (klasių svoriai)", model_b)]:
    p = model.predict(test_ds, verbose=0).ravel()
    probs[name] = p
    pred = (p >= 0.5).astype(int)
    rows.append({"Modelis": name, "Accuracy": accuracy_score(y_test, pred), "AUC": roc_auc_score(y_test, p),
                 "Precision": precision_score(y_test, pred, zero_division=0), "Recall": recall_score(y_test, pred)})
results = pd.DataFrame(rows).set_index("Modelis").round(3)
display(results)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, p) in zip(axes, probs.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, (p >= 0.5).astype(int), display_labels=["no", "yes"], ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 10 žingsnis – modelio išsaugojimas, įkėlimas ir prognozė naujiems klientams (kaip originale)

In [ ]:
SAMPLES = {
    "Klientas A (anksčiau sutiko, skambinta spalį)": {
        "age": 35, "job": "management", "marital": "single", "education": "tertiary", "default": "no",
        "balance": 1500, "housing": "no", "loan": "no", "contact": "cellular", "day": 15, "month": "oct",
        "campaign": 1, "pdays": 180, "previous": 2, "poutcome": "success"},
    "Klientas B (pirmas kontaktas, skambinta gegužę, 5-as skambutis)": {
        "age": 45, "job": "blue-collar", "marital": "married", "education": "primary", "default": "no",
        "balance": 100, "housing": "yes", "loan": "yes", "contact": "unknown", "day": 20, "month": "may",
        "campaign": 5, "pdays": -1, "previous": 0, "poutcome": "unknown"},
}

#@title 💾 STEP 10: Išsaugojimas, įkėlimas ir prognozė
model_b.save("bank_marketing_nn.keras")
reloaded_model = tf.keras.models.load_model("bank_marketing_nn.keras")

for name, sample in SAMPLES.items():
    input_dict = {k: tf.constant([[v]], dtype=tf.string if isinstance(v, str) else tf.float32) for k, v in sample.items()}
    prob = float(reloaded_model.predict(input_dict, verbose=0)[0][0])
    print(f"{name}: tikimybė sutikti su indėliu {100 * prob:.1f} %")

## Išvados
- Originalo struktūra (tf.data → Keras preprocessing sluoksniai → Dense tinklas) be pakeitimų tinka kitiems lentelės tipo verslo duomenims – keičiasi tik stulpelių sąrašai.
- Su nesubalansuotomis klasėmis accuracy yra klaidinanti: modelis A turi aukštą accuracy, bet randa mažai „yes“ klientų. Klasių svoriai (B) gerokai padidina recall, šiek tiek sumažindami accuracy ir precision.
- AUC nepriklauso nuo slenksčio, todėl juo patogiausia lyginti modelius tarpusavyje ir su Decision Forests notebook'u (ta pati testavimo aibė).
- Pašalinus `duration` rezultatai kuklesni, bet sąžiningi – modelį galima naudoti prieš skambutį.